# Week 1 — Lab
## End-to-end EDA on an LLM evaluation results table

In this lab we take a moderately sized table of LLM benchmark results and answer four
real questions a researcher might ask of it:

1. How do **accuracy distributions** differ between model families and between benchmark
   types?
2. Does **parameter count** predict accuracy, and if so, how cleanly?
3. Are some models **good on average but unreliable**? Is variance correlated with size?
4. How would we **honestly report** "method X beats method Y" if we only had three seeds?

The dataset is generated by `data/make_dataset.py` and stored as
`data/llm_eval_results.csv`. Generate it once with:

```bash
python data/make_dataset.py
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(0)

df = pd.read_csv("../data/llm_eval_results.csv")
print(df.shape)
df.head()


In [ ]:
# A first look at the schema and value ranges
df.describe(include="all").T


### Sanity checks

A few cheap checks save hours later:

- No nulls.
- Every (model, benchmark) combination has the expected number of seeds.
- Accuracies lie in [0, 1].

In [ ]:
assert df.isna().sum().sum() == 0
seeds_per_combo = df.groupby(["model", "benchmark"]).size()
assert seeds_per_combo.nunique() == 1, "Uneven seed counts!"
assert df["accuracy"].between(0, 1).all()
print(f"All checks pass. {df['model'].nunique()} models × {df['benchmark'].nunique()} benchmarks × {seeds_per_combo.iloc[0]} seeds")


## Question 1 — How do accuracy distributions differ across families and benchmark types?

A first instinct is a grouped bar chart of mean accuracies. Resist it. We have many
groups, the distributions are not symmetric, and we care about variability. Use an ECDF
and a strip plot instead.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.ecdfplot(data=df, x="accuracy", hue="family",
             ax=axes[0], lw=2)
axes[0].set_title("Accuracy ECDF by model family")
axes[0].set_xlabel("accuracy")

sns.stripplot(data=df, x="benchmark_type", y="accuracy", hue="family",
              dodge=True, alpha=0.7, size=4, ax=axes[1])
axes[1].set_title("All runs, stratified by benchmark type")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()


**Reading the chart.** The ECDF makes the family ordering immediate — the `gpt`
family curve sits to the right of `llama` and `mistral` for most of its mass, meaning
its accuracy distribution stochastically dominates them. The strip plot on the right
adds the per-benchmark-type breakdown without averaging anything away.

What we have **not** done: a single bar of means. That chart would have hidden the fact
that `research` models score very low on `math` and `gsm8k` but acceptably on
`hellaswag`.

### A side note on legend placement

`seaborn` puts the legend inside the axes by default, which often overlaps the data.
Move it outside when you have many categories — your readers will thank you.

In [ ]:
g = sns.displot(data=df, x="accuracy", hue="family", kind="ecdf",
                col="benchmark_type", col_wrap=2, height=3.2, aspect=1.4)
g.set_titles("{col_name}")
plt.show()


**Faceted ECDFs** (Tufte calls these *small multiples*) let us compare four
benchmark types at a glance without the chart getting busy. Notice how, for `safety`,
the family ranking is much weaker than for `reasoning` — that's a real-world pattern
and an honest visualization shows it instead of hiding it under a single average.

## Question 2 — Does parameter count predict accuracy?

Parameter count spans `~0.1B` to `~1500B`. **Do not** put it on a linear axis.

In [ ]:
# Aggregate across seeds for clarity, but keep the variance estimate alive
agg = (df.groupby(["model", "family", "params_b", "benchmark_type"], as_index=False)
         .accuracy.agg(mean="mean", std="std"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (a) linear x — bad
sns.scatterplot(data=agg, x="params_b", y="mean", hue="family", style="benchmark_type",
                s=60, ax=axes[0])
axes[0].set_title("Linear x — small models in a column")
axes[0].set_xlabel("parameters (B)")
axes[0].set_ylabel("mean accuracy")

# (b) log x — good
sns.scatterplot(data=agg, x="params_b", y="mean", hue="family", style="benchmark_type",
                s=60, ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_title("Log x — relationship is visible")
axes[1].set_xlabel("parameters (B, log scale)")
axes[1].set_ylabel("mean accuracy")
sns.move_legend(axes[1], "upper left", bbox_to_anchor=(1.02, 1))
sns.move_legend(axes[0], "upper left", bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()


The right-hand chart shows roughly the expected behaviour: an approximately
linear relationship between `log(params)` and mean accuracy, with **reasoning**
benchmarks shifted down (harder) and **safety** shifted up (the metric is non-toxic
rate, which has a strong baseline).

### Adding the regression line honestly

If we fit a line on `log(params)`, we should plot the prediction band along with it, not
just the central line. `seaborn.regplot` does both.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.regplot(data=agg, x="params_b", y="mean",
            scatter_kws=dict(s=42, alpha=0.7), line_kws=dict(color="black"),
            ax=ax, logx=True, ci=95)
ax.set_xscale("log")
ax.set_xlabel("parameters (B, log scale)")
ax.set_ylabel("mean accuracy")
ax.set_title("Scaling fit with 95 % CI band")
plt.show()


The band is the **uncertainty about the regression line**, not the spread of the
data. A reader who only sees the line will believe the trend is more precisely
estimated than it is.

## Question 3 — Are some models good on average but unreliable?

We define **unreliability** as the across-seed standard deviation. A useful chart pairs
the mean against the std, with model size as a third encoded variable.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(agg["mean"], agg["std"],
                s=20 + 10 * np.log10(agg["params_b"] * 1000),  # size encodes log(params)
                c=np.log10(agg["params_b"]), cmap="cividis", alpha=0.8,
                edgecolor="white", linewidth=0.5)
ax.set_xlabel("mean accuracy across seeds")
ax.set_ylabel("std across seeds")
ax.set_title("Reliability vs. capability\n(point size and colour ∝ log parameters)")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("log10(params, B)")
plt.show()


Two patterns to look for:

- **Top-right** points are capable but inconsistent — they pass a benchmark on one seed
  and fail on another. A surprising fraction of reported "SOTA" results live here.
- **Bottom-left** points are reliably bad — the model is just not capable on this task.

A method that reports only the best seed of three is hiding the top-right pattern.

## Question 4 — Honest reporting with three seeds

Suppose someone claims that the `gpt` family beats the `llama` family on `gsm8k`. With
five seeds per (model, benchmark) we can do a cheap **bootstrap** to give an honest
interval.

In [ ]:
def bootstrap_mean_ci(x, n_boot=5000, alpha=0.05, rng=RNG):
    x = np.asarray(x)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    means = x[idx].mean(axis=1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return float(x.mean()), float(lo), float(hi)


target = df[df["benchmark"] == "gsm8k"]
rows = []
for fam, sub in target.groupby("family"):
    m, lo, hi = bootstrap_mean_ci(sub["accuracy"].values)
    rows.append(dict(family=fam, mean=m, lo=lo, hi=hi, n=len(sub)))
boot = pd.DataFrame(rows).sort_values("mean")
boot


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
y = np.arange(len(boot))
ax.hlines(y, boot["lo"], boot["hi"], color="black", lw=2)
ax.plot(boot["mean"], y, "o", color="#0072B2", markersize=8)
ax.set_yticks(y, boot["family"])
ax.set_xlim(0, 1)
ax.set_xlabel("mean accuracy on gsm8k (95 % bootstrap CI)")
ax.set_title("Family-level comparison, with uncertainty")
plt.show()


This chart says exactly what the data supports: families are ordered, but the
intervals overlap. Reporting only the mean would obscure that.

> **Style note.** When the question is "which interval is bigger?", *position on a
> common scale* (here: x-axis) is the right encoding. A horizontal dot-plot with error
> bars wins over a vertical grouped bar with whiskers basically every time.

## What to do differently in your own research

- Don't aggregate before you've at least once looked at the per-seed strip plot. If
  variance is comparable to your effect, you may not have an effect.
- Default to ECDFs for cross-group distribution comparisons. They are robust to
  bandwidth choice, easy to overlay, and read off as quantiles directly.
- Spans of orders of magnitude are everywhere in AI. Decide your scale **before** you
  pick your chart type — a wrong scale invalidates the choice that follows.
- The bootstrap is cheap and assumption-light. There is rarely a good reason not to
  report a CI alongside a mean.

### Where to go next

Open `exercises/01-distributions.ipynb` for three open-ended problems on this dataset.
Solutions are in `solutions/`.
